In [ ]:
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
movies = pd.read_csv("dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("dataset/tmdb_5000_credits.csv")

In [ ]:
movies = movies.merge(credits, on="title")

print("Dataset merged successfully!")
print(movies.shape)

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

movies.head()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

print(movies.shape)

In [ ]:
def convert(obj):
    L = []
    
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    
    return L

movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
def convert3(obj):
    L = []
    counter = 0
    
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    
    return L

movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
def fetch_director(obj):
    L = []
    
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    
    return L

movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['tags'] = (
    movies['overview'] + 
    movies['genres'].apply(lambda x: ' '.join(x)) + 
    movies['keywords'].apply(lambda x: ' '.join(x)) + 
    movies['cast'].apply(lambda x: ' '.join(x)) + 
    movies['crew'].apply(lambda x: ' '.join(x))
)

movies[['title', 'tags']].head()

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(movies['tags']).toarray()

print("Text converted into numbers!")
print(vectors.shape)

In [ ]:
similarity = cosine_similarity(vectors)

print("Similarity calculated successfully!")
print(similarity.shape)

In [ ]:
def recommend(movie):
    movie_index = movies[movies['title'] == movie].index[0]
    
    distances = similarity[movie_index]
    
    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]
    
    print("Recommended movies for:", movie)
    
    for i in movie_list:
        print(movies.iloc[i[0]].title)

In [31]:
recommend("Superman")

Recommended movies for: Superman
Superman II
Man of Steel
Superman Returns
Superman IV: The Quest for Peace
X-Men: Apocalypse
